# Lesson 2: Sequential Chats and Customer Onboarding

## Setup

In [11]:
llm_config={"model": "gpt-3.5-turbo"}

In [12]:
from utils import get_openai_api_key
OPENAI_API_KEY = get_openai_api_key()

In [13]:
from autogen import ConversableAgent

## Creating the needed agents

In [15]:
onboarding_personal_information_agent=ConversableAgent(
    name="Onboarding_Personal_Information_Agent",
    system_message='''You are a helpful customer onboarding agent,
    you are here to help new customers get started with our product.
    Your job is to gather customer's name and location.
    Do not ask for other information. Return 'TERMINATE' 
    when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [16]:
onboarding_topic_preference_agent = ConversableAgent(
    name="Onboarding_Topic_preference_Agent",
    system_message='''You are a helpful customer onboarding agent,
    you are here to help new customers get started with our product.
    Your job is to gather customer's preferences on news topics.
    Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [17]:
customer_engagement_agent = ConversableAgent(
    name="Customer_Engagement_Agent",
    system_message='''You are a helpful customer service agent
    here to provide fun for the customer based on the user's
    personal information and topic preferences.
    This could include fun facts, jokes, or interesting stories.
    Make sure to make it engaging and fun!
    Return 'TERMINATE' when you are done.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)

In [18]:
customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)

## Creating tasks

Now, you can craft a series of tasks to facilitate the onboarding process.

In [19]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": 
            "Hello, I'm here to help you get started with our product."
            "Could you tell me your name and location?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt" : "Return the customer information "
                             "into as JSON object only: "
                             "{'name': '', 'location': ''}",
        },
        "max_turns": 2,
        "clear_history" : True
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message": 
                "Great! Could you tell me what topics you are "
                "interested in reading about?",
        "summary_method": "reflection_with_llm",
        "max_turns": 1,
        "clear_history" : False
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1,
        "summary_method": "reflection_with_llm",
    },
]

## Start the onboarding process

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different inputs, such as name, location, and preferences.

In [20]:
from autogen import initiate_chats

chat_results = initiate_chats(chats)


********************************************************************************
Starting a new chat....

********************************************************************************
Onboarding_Personal_Information_Agent (to customer_proxy_agent):

Hello, I'm here to help you get started with our product.Could you tell me your name and location?

--------------------------------------------------------------------------------


/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/agents/agents_venv/lib/python3.13/site-packages/autogen/agentchat/chat.py:69: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(


Replying as customer_proxy_agent. Provide feedback to Onboarding_Personal_Information_Agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Ola


customer_proxy_agent (to Onboarding_Personal_Information_Agent):

Ola

--------------------------------------------------------------------------------
Onboarding_Personal_Information_Agent (to customer_proxy_agent):

I'm here to help you get started with our product. Could you please provide me with your name and location?

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to Onboarding_Personal_Information_Agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Chicago


customer_proxy_agent (to Onboarding_Personal_Information_Agent):

Chicago

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (320cf1e9-5c04-4af7-a89d-923a3b53179f): Maximum turns (2) reached

********************************************************************************
Starting a new chat....

********************************************************************************
Onboarding_Topic_preference_Agent (to customer_proxy_agent):

Great! Could you tell me what topics you are interested in reading about?
Context: 
{'name': 'Ola', 'location': 'Chicago'}

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to Onboarding_Topic_preference_Agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  AI Agents


customer_proxy_agent (to Onboarding_Topic_preference_Agent):

AI Agents

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (13836c8b-0e6c-4de2-9ab7-77d157b572d3): Maximum turns (1) reached

********************************************************************************
Starting a new chat....

********************************************************************************
customer_proxy_agent (to Customer_Engagement_Agent):

Let's find something fun to read.
Context: 
{'name': 'Ola', 'location': 'Chicago'}
Ola from Chicago is interested in reading about AI Agents.

--------------------------------------------------------------------------------
Customer_Engagement_Agent (to customer_proxy_agent):

Hey Ola from Chicago! Did you know that the concept of AI agents has been around for decades? One of the earliest examples of an AI agent is ELIZA, created in the 1960s. ELIZA was a computer program that simulated a conversation with 

## Print out the summary

In [21]:
for chat_result in chat_results:
    print(chat_result.summary)
    print("\n")

{'name': 'Ola', 'location': 'Chicago'}


Ola from Chicago is interested in reading about AI Agents.


Ola from Chicago is interested in reading about AI Agents, which have evolved significantly since the creation of early examples like ELIZA in the 1960s.




## Print out the cost

In [22]:
for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0.00017, 'gpt-3.5-turbo-0125': {'cost': 0.00017, 'prompt_tokens': 232, 'completion_tokens': 36, 'total_tokens': 268}}, 'usage_excluding_cached_inference': {'total_cost': 0.00017, 'gpt-3.5-turbo-0125': {'cost': 0.00017, 'prompt_tokens': 232, 'completion_tokens': 36, 'total_tokens': 268}}}


{'usage_including_cached_inference': {'total_cost': 5.6e-05, 'gpt-3.5-turbo-0125': {'cost': 5.6e-05, 'prompt_tokens': 76, 'completion_tokens': 12, 'total_tokens': 88}}, 'usage_excluding_cached_inference': {'total_cost': 5.6e-05, 'gpt-3.5-turbo-0125': {'cost': 5.6e-05, 'prompt_tokens': 76, 'completion_tokens': 12, 'total_tokens': 88}}}


{'usage_including_cached_inference': {'total_cost': 0.000324, 'gpt-3.5-turbo-0125': {'cost': 0.000324, 'prompt_tokens': 282, 'completion_tokens': 122, 'total_tokens': 404}}, 'usage_excluding_cached_inference': {'total_cost': 0.000324, 'gpt-3.5-turbo-0125': {'cost': 0.000324, 'prompt_tokens': 282, 'completion_tokens':